In [1]:
!pip install kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download("rmisra/news-category-dataset")

print(path)

100%|██████████| 26.5M/26.5M [00:00<00:00, 46.1MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/rmisra/news-category-dataset/versions/3


In [3]:
import os
import pandas as pd

print(os.listdir(path))

['News_Category_Dataset_v3.json']


In [4]:
df = pd.read_json(os.path.join(path, "News_Category_Dataset_v3.json"), lines=True)

df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [6]:
import re
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
df["text"] = df["headline"] + " " + df["short_description"]

In [8]:
df = df.dropna(subset=["text","category"])

In [9]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [10]:
df["clean_text"] = df["text"].apply(clean_text)

In [11]:
df[["text","clean_text"]].head()

,text,clean_text
0,Over 4 Million Americans Roll Up Sleeves For O...,million americans roll sleeves omicrontargeted...
1,"American Airlines Flyer Charged, Banned For Li...",american airlines flyer charged banned life pu...
2,23 Of The Funniest Tweets About Cats And Dogs ...,funniest tweets cats dogs week sept dog dont u...
3,The Funniest Tweets From Parents This Week (Se...,funniest tweets parents week sept accidentally...
4,Woman Who Called Cops On Black Bird-Watcher Lo...,woman called cops black birdwatcher loses laws...


In [12]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]

y = df["category"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [14]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [15]:
prediction = model.predict(X_test_tfidf)

prediction

array(['WELLNESS', 'BUSINESS', 'POLITICS', ..., 'HOME & LIVING', 'TRAVEL',
       'POLITICS'], dtype='<U14')

In [19]:
news = """
Australia defeated India  by 8 wickets in the final match.
Shane Watson scored a brilliant century and won Player of the Match.
"""

news_clean = clean_text(news)

news_vector = tfidf.transform([news_clean])

result = model.predict(news_vector)

print("Predicted Category :", result[0])


Predicted Category : SPORTS


In [20]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test,prediction)

print("Accuracy :",accuracy)

Accuracy : 0.5202596286927886


In [21]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,prediction)

print(cm)

[[  7   0   1 ...   2   1   1]
 [  1   2   2 ...   2   0   0]
 [  0   0  72 ...   1   0   0]
 ...
 [  0   0   4 ...  61   0   0]
 [  0   0   0 ...   0 101   3]
 [  0   0   2 ...   0  14  28]]
